# 23 · Prototype backend — trả lời giống eval không? Độ tin cậy có ý nghĩa không? (W13, R-PAR · R-CONF · R-YB-det)

**Vấn đề:** prototype phải trả lời **giống hệt** pipeline đã báo cáo, nếu không thì số liệu khoá luận không còn đúng cho bản demo. Ngoài ra, con số "độ tin cậy" hiện trên giao diện phải phản ánh đúng khả năng trả lời đúng, và bộ nhận diện câu có/không (sửa thiên lệch "có" của SH, R-YB v3) không được làm hỏng câu hỏi thường.

**Notebook này chạy backend prototype (`src/prototype/core.py`, adapter SH) trên:**
- ViVQA test (3.001 câu) và ViTextVQA test2000 (2.000 câu): greedy y hệt eval, kèm log-prob từng token.
- POPE-vi mẫu câu 3 (`Có X trong ảnh không? Trả lời "có" hoặc "không".`, 4.985 câu): chỉ mô tả.

**Luật ghi trước** (`docs/W13/README.md`):
| Luật | ĐẠT khi |
|---|---|
| **R-PAR** | câu trả lời trùng hệt run eval W11 ≥ 99% ở từng tập |
| **R-CONF** | trên nửa B (theo ảnh), EM giảm ngặt Cao > Trung bình > Thấp, cả gộp lẫn từng tập (ngưỡng chọn trên nửa A) |
| **R-YB-det** | câu bị detector bắt đi đường có/không làm EM toàn tập giảm ≤ 1,5 điểm ở từng tập |

| Mục | Việc | Thời gian |
|---|---|---|
| 1–2 | thiết lập, kiểm tra, selftest | ≈ 10 phút |
| 3 | **smoke parity** 100 câu: trùng eval ≥ 99% mới chạy tiếp | ≈ 3 phút |
| 4 | ViVQA test → ViTextVQA test2000 → POPE mẫu 3 | ≈ 1,5 h (chép ảnh ViTextVQA ≈ 10 phút) |
| 5–6 | phân tích + phán quyết, tự trả máy | ≈ 1 phút |

## 1. Thiết lập môi trường

Mount Drive, lấy code mới nhất từ GitHub, khai báo đường dẫn. Notebook chạy **toàn bộ trên L4**.

*Kỳ vọng: commit mới nhất (`W13 step 4-6 …`), `Runtime: NVIDIA L4`, `✅ Thiết lập xong`.*

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, hashlib, subprocess
if not os.path.exists('/content/ViVQA-VLM/.git'):
    !cd /content && git clone -q https://github.com/PLHGNAOH/ViVQA-VLM.git
%cd /content/ViVQA-VLM
!git pull -q origin main
!git log -1 --oneline
sys.path.insert(0, '/content/ViVQA-VLM')
from src.eval.prototype_check import CHECK_VERSION
assert CHECK_VERSION == 'prototype-check-v1', 'Code cũ -> git pull chưa lấy được commit bước 4-6'

DRIVE_ROOT = '/content/drive/MyDrive/ViVQA-VLM'
EXP_ROOT = f'{DRIVE_ROOT}/experiments'
OUT = f'{EXP_ROOT}/W13_prototype_check_L4_seed42'
T3_REPO = 'experiments/W13_yesno_questions/pope_vi_questions_t3_v1.json'
T3_SHA = 'd03600fc5f7a1590f5ed44d45cd0a4429eba0bcecd328390d6af6a182db4ca8d'

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''):
            h.update(b)
    return h.hexdigest()

def run_cmd(cmd, check=True):
    # Chạy lệnh, in log trực tiếp; lỗi -> DỪNG notebook (trừ khi check=False: trả mã lỗi)
    print('$', cmd)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='')
    rc = p.wait()
    if check and rc != 0:
        raise RuntimeError(f'Lệnh lỗi (mã {rc}) -> xem log ở trên, gửi Claude')
    return rc

GPU = subprocess.run('nvidia-smi --query-gpu=name --format=csv,noheader', shell=True,
                     capture_output=True, text=True).stdout.strip() or 'CPU'
print('Runtime:', GPU)
print('✅ Thiết lập xong')

## 2. Kiểm tra trước khi tốn GPU

Phần này **bắt buộc L4**, cùng GPU với run eval W11. Nếu đang ở runtime khác thì vào **Runtime → Change runtime type → L4 → Save**, rồi **Runtime → Run all**.

- **(a)** Cài thư viện, cùng pin `transformers==5.17.0`.
- **(b)** Cấu hình OCR đọc từ `ocr_selection.json` (W08).
- **(c)** Hai run eval W11 của SH (mốc parity) có trên Drive; adapter SH trên Drive trùng dấu vân tay với run W11.
- **(d)** File mẫu câu 3 trong repo đúng sha256 đã chốt.
- **(e)** Selftest `prototype_check` (có phép so `generate` + `output_scores` với `generate` của eval trên Qwen2.5-VL tí hon), `yesno_calib`, `pope_vi`.

Phép kiểm quan trọng nhất (chữ ký cấu hình trùng W11) chạy **bên trong** mỗi lượt chấm ở mục 3–4: lệch là dừng ngay.

*Kỳ vọng: `GPU: NVIDIA L4`, `paddledet_vietocr | v1 | 600 | 0.0`, 2 dòng `mốc W11 … ✅`, `adapter SH = run W11 ✅`, `mẫu câu 3 = bản chốt ✅`, `✅ selftest OK (3 script)`.*

In [ ]:
if 'L4' not in GPU:
    raise SystemExit('❌ Notebook 23 phải chạy trên L4 (cùng GPU với run eval W11). '
                     'Runtime -> Change runtime type -> L4, rồi Run all.')
print('GPU:', GPU)
!pip install -q transformers==5.17.0 accelerate qwen-vl-utils bitsandbytes peft huggingface_hub pyyaml

from src.eval.run_baseline import adapter_fingerprint
from src.eval.prototype_check import REF_RUN

# (b) OCR: cấu hình đóng băng W08
sel = json.load(open('experiments/W08_ocr_select_dev300_L4_seed42/ocr_selection.json', encoding='utf-8'))['decision']
BACKEND, TPL, MAXC, MINC = sel['backend'], sel['template'], sel['max_chars'], sel['min_conf']
print(BACKEND, '|', TPL, '|', MAXC, '|', MINC)
OCR_ARGS = f'--ocr_backend {BACKEND} --ocr_template {TPL} --ocr_max_chars {MAXC} --ocr_min_conf {MINC}'

# (c) mốc W11 + adapter SH
for ds, run in REF_RUN.items():
    for f in ('metrics.json', 'predictions.json', 'run_signature.json'):
        assert os.path.exists(f'{EXP_ROOT}/{run}/{f}'), f'thiếu {run}/{f} trên Drive -> báo Claude'
    print(f'mốc W11 {ds}: {run} ✅')
ref = json.load(open(f"{EXP_ROOT}/{REF_RUN['vivqa']}/metrics.json", encoding='utf-8'))['meta']['adapter']
fp = adapter_fingerprint(ref['path'])
assert fp['weights_sha256'] == ref['weights_sha256'], 'adapter SH trên Drive KHÁC run W11 -> dừng, báo Claude'
AD_SH = ref['path']
print(f"adapter SH = run W11 ✅ ({fp['weights_mb']} MB, {fp['weights_sha256'][:12]}…)")

# (d) mẫu câu 3
assert sha256_file(T3_REPO) == T3_SHA, 'mẫu câu 3 trong repo KHÁC bản chốt -> báo Claude'
print('mẫu câu 3 = bản chốt ✅')

# (e) selftest
musts = {'src.eval.prototype_check': ['prototype_check selftest OK', 'tiny Qwen2.5-VL OK'],
         'src.eval.yesno_calib': ['yesno_calib selftest OK', 'tiny Qwen2.5-VL OK'],
         'src.eval.pope_vi': ['pope_vi selftest OK']}
for mod, need in musts.items():
    o = subprocess.run(f'python -m {mod} --selftest', shell=True, capture_output=True, text=True)
    for m in need:
        assert m in o.stdout, (mod, m, o.stdout[-800:], o.stderr[-800:])
print('✅ selftest OK (3 script)')

## 3. Smoke parity: backend có trả lời trùng hệt eval W11 không? (≈ 3 phút)

Chạy backend trên **100 câu ViVQA test đầu tiên**, rồi so từng câu trả lời với run eval W11. Trước khi chấm, code tự kiểm **chữ ký cấu hình** (model, 4-bit, prompt, OCR, dữ liệu, adapter) dựng lại phải trùng hệt run W11. Bước này cũng chép ảnh ViVQA về đĩa Colab (≈ 5 phút lần đầu).

*Kỳ vọng: `chữ ký cấu hình = W11 (c12787d15649) ✅`, `trùng hệt run W11: ≥ 99% …` và `✅ … chạy tiếp`. Nếu `❌` hoặc lỗi chữ ký → notebook DỪNG; gửi Claude output cell này.*

In [ ]:
from src.eval.prototype_check import SMOKE_RUN
if not os.path.exists(f'{EXP_ROOT}/{SMOKE_RUN}/records.json'):
    run_cmd(f'python -u -m src.eval.prototype_check --score --dataset vivqa --run_name {SMOKE_RUN} '
            f'--drive_root {DRIVE_ROOT} --limit 100 --chunk 100 {OCR_ARGS}')
m = json.load(open(f'{EXP_ROOT}/{SMOKE_RUN}/meta.json', encoding='utf-8'))
print(f"s/câu: {m['sec_per_sample']} | VRAM đỉnh: {m.get('peak_vram_gb')} GB | repetition_penalty: {m['repetition_penalty']}")
rc = run_cmd(f"python -m src.eval.prototype_check --parity --run_dir {EXP_ROOT}/{SMOKE_RUN} "
             f"--ref_dir {EXP_ROOT}/{REF_RUN['vivqa']}", check=False)
if rc != 0:
    raise SystemExit('⏸ DỪNG: backend chưa tái hiện eval -> gửi Claude output cell này (không chạy tiếp).')

## 4. Chạy 3 lượt (≈ 1,5 giờ)

| Thứ tự | Lượt | Câu | Thời gian |
|---|---|---|---|
| 1 | Backend trên ViVQA test | 3.001 | ≈ 30 phút |
| 2 | Backend trên ViTextVQA test2000 (lần đầu chép ảnh ViTextVQA ≈ 10 phút) | 2.000 | ≈ 45 phút |
| 3 | SH + τ* trên POPE-vi mẫu câu 3 (chấm margin như R-YB v3) | 4.985 | ≈ 20 phút |

Mỗi lô ghi lên Drive ngay khi xong. **Bị ngắt thì Run all lại**: lượt đã xong tự bỏ qua, lượt dở chạy tiếp từ lô cuối. Giữ máy thức và tab mở.

*Kỳ vọng: mỗi lượt 1–2 in `chữ ký cấu hình = W11 (…) ✅` rồi các dòng lô, cuối cùng `== xong … câu | EM …` (EM phải gần EM của W11: ViVQA ≈ 73,51, ViTextVQA ≈ 49,50). Lượt 3 in `ứng viên … repetition_penalty … = 1.05` rồi `== xong 4985 câu`.*

In [ ]:
from src.eval.prototype_check import PT_RUN, T3_RUN
for ds in ('vivqa', 'vitextvqa_official'):
    run = PT_RUN[ds]
    if os.path.exists(f'{EXP_ROOT}/{run}/records.json'):
        print(f'⏭ {ds}: đã có -> bỏ qua')
    else:
        run_cmd(f'python -u -m src.eval.prototype_check --score --dataset {ds} --run_name {run} '
                f'--drive_root {DRIVE_ROOT} --chunk 500 {OCR_ARGS}')
    m = json.load(open(f'{EXP_ROOT}/{run}/meta.json', encoding='utf-8'))
    print(f"✔ {ds}: {m['num_samples']} câu | EM {100 * m['em']:.2f} | {m['sec_per_sample']} s/câu | câu có/không bị bắt: {m['n_yesno_detected']}\n")

if os.path.exists(f'{EXP_ROOT}/{T3_RUN}/logits.json'):
    print('⏭ POPE mẫu câu 3: đã có -> bỏ qua')
else:
    run_cmd(f'python -u -m src.eval.yesno_calib --score --data_file {T3_REPO} --run_name {T3_RUN} '
            f'--drive_root {DRIVE_ROOT} --chunk 500 --mode ocr {OCR_ARGS} --adapter {AD_SH}')
m = json.load(open(f'{EXP_ROOT}/{T3_RUN}/meta.json', encoding='utf-8'))
print(f"✔ POPE mẫu câu 3: {m['num_samples']} câu | {m['sec_per_sample']} s/câu")

## 5. Phân tích + phán quyết (R-PAR · R-CONF · R-YB-det)

`src.eval.prototype_check --analyze`:
1. **R-PAR**: % câu trả lời trùng hệt run W11 ở từng tập, liệt kê câu lệch.
2. **R-CONF**: ngưỡng Thấp/Trung bình/Cao chọn trên nửa A (theo ảnh), EM theo mức trên nửa B, ECE.
3. **R-YB-det**: câu bị detector bắt → EM toàn tập giảm bao nhiêu nếu đi đường có/không.
4. **Chỉ mô tả**: độ tin cậy câu có/không trên POPE-vi; τ* với mẫu câu 3; chi phí.

*Kỳ vọng: 6 mục, 3 dòng `➜ **R-PAR: …**`, `➜ **R-CONF: …**`, `➜ **R-YB-det: …**`. File `prototype_check_summary.json` + `.md` nằm trong `experiments/W13_prototype_check_L4_seed42/`.*

In [ ]:
os.makedirs(OUT, exist_ok=True)
run_cmd(f'python -m src.eval.prototype_check --analyze --exp_root {EXP_ROOT} --out {OUT}')

## 6. Tự trả máy L4 khi chạy xong

Kết quả đã nằm trên Drive. Output phía trên vẫn hiện trong tab sau khi trả máy. Nếu notebook dừng sớm vì lỗi thì máy **không** tự trả: đọc lỗi xong, tự bấm **Runtime → Disconnect and delete runtime**.

*Kỳ vọng: `✅ Xong toàn bộ -> tự trả máy …`.*

In [ ]:
AUTO_RELEASE = True
if AUTO_RELEASE:
    print('✅ Xong toàn bộ -> tự trả máy để ngừng tính CU (output phía trên vẫn giữ trong tab).')
    from google.colab import runtime
    runtime.unassign()
else:
    print('AUTO_RELEASE = False -> nhớ tự Disconnect and delete runtime khi xong.')

## 7. ⏸ DỪNG

Nhắn Claude **"xong"** + ảnh CU (Claude tự đọc kết quả trên Drive), hoặc gửi output **mục 3** và **mục 5**. Kết quả nằm trong `experiments/W13_PT_*`, `W13_YN3_…_popevi_t3_…` và `W13_prototype_check_L4_seed42/` trên Drive.